<a href="https://colab.research.google.com/github/chal0326/researchcms/blob/main/epsteinneo4j.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install neo4j spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 125.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
!pip install sentence-transformers pandas
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
# Download and install cloudflared
!curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared.deb

# Run the bridge in the background
# This maps 'localhost:7687' in Kaggle -> 'bolt.ahhhh.men' -> Your Local DB
import subprocess

# REPLACE THESE with your Service Token if Access is enabled.
# If Access is disabled/bypassed, you can remove the --id and --secret flags.
cmd = [
    "cloudflared", "access", "tcp",
    "--hostname", "bolt.ahhhh.men",
    "--url", "localhost:7687",
    "--id", "facec3bfaf0d896f095f368020071d7a.access",      # Remove if Access is disabled
    "--secret", "7478557ddccc780cd3322528be98e3f8e811c63ab774cf0f239985d89a2c988a" # Remove if Access is disabled
]

# Run as a background process
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Bridge started on localhost:7687")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 19.1M  100 19.1M    0     0  17.8M      0  0:00:01  0:00:01 --:--:-- 46.6M
(Reading database ... 121693 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.1.2) over (2026.1.2) ...
Setting up cloudflared (2026.1.2) ...
Processing triggers for man-db (2.10.2-1) ...
Bridge started on localhost:7687


In [ ]:
# Install Neo4j and Sentence Transformers if not already there
!pip install neo4j sentence-transformers

import pandas as pd
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer, util

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "jaspeR154$")

try:
    driver = GraphDatabase.driver(URI, auth=AUTH)
    driver.verify_connectivity()
    print("✅ Connection Successful!")

except Exception as e:
    print(f"❌ Connection Failed: {e}")


✅ Connection Successful!


In [ ]:
import spacy
from neo4j import GraphDatabase
import tqdm # For a progress bar

# Load SpaCy model
nlp = spacy.load("en_core_web_sm")

# Connection Config (Already verified by your previous cell)
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "jaspeR154$")

def run_extraction():
    driver = GraphDatabase.driver(URI, auth=AUTH)

    # 1. Create Constraints (Crucial for performance)
    with driver.session() as session:
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")
        print("✅ Constraints Verified.")

    # 2. Process in Batches
    batch_size = 500
    while True:
        with driver.session() as session:
            # Fetch documents that don't have a 'processed' flag yet
            docs = session.run("""
                MATCH (d:Document)
                WHERE d.processed IS NULL AND d.content IS NOT NULL
                RETURN d.doc_name AS name, d.content AS text
                LIMIT $limit
            """, limit=batch_size)

            data_list = list(docs)
            if not data_list:
                print("🏁 All documents processed!")
                break

            payload = []
            for record in tqdm.tqdm(data_list, desc="Processing Entities"):
                doc_name = record["name"]
                text = record["text"]

                # NLP Magic
                spacy_doc = nlp(text[:50000]) # Cap text per file to avoid memory spikes
                entities = [{"name": ent.text, "type": ent.label_}
                            for ent in spacy_doc.ents
                            if ent.label_ in ["PERSON", "ORG"]]

                payload.append({"doc_name": doc_name, "entities": entities})

            # 3. Batch Write back to the Pi
            session.run("""
                UNWIND $payload AS item
                MATCH (d:Document {doc_name: item.doc_name})
                SET d.processed = true
                WITH d, item
                UNWIND item.entities AS ent
                MERGE (e:Entity {name: ent.name})
                ON CREATE SET e.type = ent.type
                MERGE (d)-[:MENTIONS]->(e)
            """, payload=payload)

    driver.close()

run_extraction()

✅ Constraints Verified.


Processing Entities: 100%|██████████| 500/500 [00:21<00:00, 22.93it/s]
ERROR:neo4j.io:[#C788]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687))): OSError('No data')


KeyboardInterrupt: 

# Task
Update the `run_extraction` function in cell `_lxtAokPILA8` to use a Hugging Face Named Entity Recognition (NER) model instead of SpaCy for extracting 'PERSON' and 'ORG' entities from documents, then run the modified function to process documents and save the extracted entities to Neo4j.

## Prepare Hugging Face Environment

### Subtask:
Ensure all necessary Hugging Face libraries are installed and ready for use. This might involve additional `pip install` commands if needed, although `sentence-transformers` already brings in a lot of `transformers` dependencies.


**Reasoning**:
The subtask requires importing the `pipeline` function from the `transformers` library. I will use a code block to perform this import.



In [ ]:
from transformers import pipeline
print("Imported pipeline from transformers")

**Reasoning**:
Now that the `pipeline` function is imported, the next step is to initialize the Named Entity Recognition (NER) pipeline from Hugging Face. This will load a pre-trained NER model which will be used to extract entities from the text.



In [ ]:
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER")
print("Hugging Face NER pipeline initialized.")

NameError: name 'pipeline' is not defined

**Reasoning**:
The previous attempt to initialize the NER pipeline failed because the `pipeline` function was not defined in the current scope, indicating that the import statement was not executed. To resolve this, I will modify the failing cell (`43929f61`) to include the `pipeline` import directly, ensuring it is available when `ner_pipeline` is initialized.



In [ ]:
from transformers import pipeline
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER")
print("Hugging Face NER pipeline initialized.")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]


KeyboardInterrupt



# Task
`ner_pipeline` was not successfully initialized in the previous step due to a `KeyboardInterrupt`. I will re-run the cell `827aa0e2` to ensure the Hugging Face NER model is loaded before proceeding.

```python
from transformers import pipeline
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER")
print("Hugging Face NER pipeline initialized.")
```

## Load Hugging Face NER Model

### Subtask:
Load a pre-trained Named Entity Recognition (NER) model from Hugging Face using the `pipeline` function. We'll specifically choose a model capable of identifying 'PERSON' and 'ORG' entities, such as `dslim/bert-base-NER`.


**Reasoning**:
To load the pre-trained Named Entity Recognition (NER) model, I will re-run the code in cell `827aa0e2` to ensure the `pipeline` function is imported and the `ner_pipeline` is correctly initialized with the 'dslim/bert-base-NER' model.



In [ ]:
from transformers import pipeline
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER")
print("Hugging Face NER pipeline initialized.")

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


Hugging Face NER pipeline initialized.


**Reasoning**:
The previous step successfully initialized the Hugging Face NER pipeline. Now, I will update the `run_extraction` function in cell `_lxtAokPILA8` to use this `ner_pipeline` instead of SpaCy for entity extraction and then execute the function to process the documents.



In [ ]:
from neo4j import GraphDatabase
import tqdm # For a progress bar

# Assume ner_pipeline is globally available from previous cell execution
# URI and AUTH are also globally available

# Helper function to process Hugging Face NER output
def process_hf_ner_output(ner_results):
    processed_entities = []
    current_entity = {}

    for ent in ner_results:
        # Extract base entity type (e.g., PER, ORG) from tags like 'B-PER', 'I-ORG'
        entity_type_raw = ent['entity'][2:]
        prefix = ent['entity'][:2]

        # Handle '##' subword tokens by removing them for cleaner entity names
        word_text = ent['word'].replace('##', '')

        if prefix == 'B-':
            if current_entity: # If there was a previous entity, add it to the list
                processed_entities.append(current_entity)
            current_entity = {"name": word_text, "type": entity_type_raw}
        elif prefix == 'I-':
            # Extend current entity if it exists and type matches
            if current_entity and current_entity["type"] == entity_type_raw:
                current_entity["name"] += " " + word_text
            else: # If I- without B- or type mismatch, start a new entity for robustness
                if current_entity:
                    processed_entities.append(current_entity)
                current_entity = {"name": word_text, "type": entity_type_raw}
        else: # Handle 'O' (outside) tag or other unexpected prefixes by ending current entity
            if current_entity:
                processed_entities.append(current_entity)
            current_entity = {} # Reset current entity

    if current_entity: # Add the last entity if it exists after the loop
        processed_entities.append(current_entity)

    # Filter for PERSON and ORG and normalize type names to match original SpaCy output types
    final_entities = []
    for entity in processed_entities:
        if entity["type"] == "PER":
            final_entities.append({"name": entity["name"], "type": "PERSON"})
        elif entity["type"] == "ORG":
            final_entities.append({"name": entity["name"], "type": "ORG"})
    return final_entities

def run_extraction():
    driver = GraphDatabase.driver(URI, auth=AUTH)

    # 1. Create Constraints (Crucial for performance)
    with driver.session() as session:
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")
        print("✅ Constraints Verified.")

    # 2. Process in Batches
    batch_size = 500
    while True:
        with driver.session() as session:
            # Fetch documents that don't have a 'processed' flag yet
            docs = session.run("""
                MATCH (d:Document)
                WHERE d.processed IS NULL AND d.content IS NOT NULL
                RETURN d.doc_name AS name, d.content AS text
                LIMIT $limit
            """, limit=batch_size)

            data_list = list(docs)
            if not data_list:
                print("🏁 All documents processed!")
                break

            payload = []
            for record in tqdm.tqdm(data_list, desc="Processing Entities"):
                doc_name = record["name"]
                text = record["text"]

                # Hugging Face NER Magic
                # Cap text per file to avoid memory spikes. The Hugging Face pipeline
                # typically handles chunking for models with max_sequence_length internally.
                chunk_text = text[:50000] # General cap to limit input size
                ner_results = ner_pipeline(chunk_text)

                entities = process_hf_ner_output(ner_results)

                payload.append({"doc_name": doc_name, "entities": entities})

            # 3. Batch Write back to the Neo4j
            session.run("""
                UNWIND $payload AS item
                MATCH (d:Document {doc_name: item.doc_name})
                SET d.processed = true
                WITH d, item
                UNWIND item.entities AS ent
                MERGE (e:Entity {name: ent.name})
                ON CREATE SET e.type = ent.type
                MERGE (d)-[:MENTIONS]->(e)
            """, payload=payload)

    driver.close()

run_extraction()


ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [Errno 111] Connection refused)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)

**Reasoning**:
The previous code failed because the connection to Neo4j via the `cloudflared` tunnel was lost. This often happens when the background process for `cloudflared` terminates. To re-establish the connection, I need to restart the `cloudflared` tunnel by re-running the cell that initializes it.



In [ ]:
# Download and install cloudflared
!curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared.deb

# Run the bridge in the background
# This maps 'localhost:7687' in Kaggle -> 'bolt.ahhhh.men' -> Your Local DB
import subprocess

# REPLACE THESE with your Service Token if Access is enabled.
# If Access is disabled/bypassed, you can remove the --id and --secret flags.
cmd = [
    "cloudflared", "access", "tcp",
    "--hostname", "bolt.ahhhh.men",
    "--url", "localhost:7687",
    "--id", "facec3bfaf0d896f095f368020071d7a.access",      # Remove if Access is disabled
    "--secret", "7478557ddccc780cd3322528be98e3f8e811c63ab774cf0f239985d89a2c988a" # Remove if Access is disabled
]

# Run as a background process
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Bridge started on localhost:7687")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 19.1M  100 19.1M    0     0  20.4M      0 --:--:-- --:--:-- --:--:-- 20.4M
(Reading database ... 121693 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.1.2) over (2026.1.2) ...
Setting up cloudflared (2026.1.2) ...
Processing triggers for man-db (2.10.2-1) ...
Bridge started on localhost:7687


**Reasoning**:
Since the `cloudflared` tunnel has been re-established, I will now re-execute the `run_extraction` function, which has already been updated to use the Hugging Face NER pipeline. This will allow it to connect to Neo4j and process the documents.



In [ ]:
from neo4j import GraphDatabase
import tqdm # For a progress bar

# Assume ner_pipeline is globally available from previous cell execution
# URI and AUTH are also globally available

# Helper function to process Hugging Face NER output
def process_hf_ner_output(ner_results):
    processed_entities = []
    current_entity = {}

    for ent in ner_results:
        # Extract base entity type (e.g., PER, ORG) from tags like 'B-PER', 'I-ORG'
        entity_type_raw = ent['entity'][2:]
        prefix = ent['entity'][:2]

        # Handle '##' subword tokens by removing them for cleaner entity names
        word_text = ent['word'].replace('##', '')

        if prefix == 'B-':
            if current_entity: # If there was a previous entity, add it to the list
                processed_entities.append(current_entity)
            current_entity = {"name": word_text, "type": entity_type_raw}
        elif prefix == 'I-':
            # Extend current entity if it exists and type matches
            if current_entity and current_entity["type"] == entity_type_raw:
                current_entity["name"] += " " + word_text
            else: # If I- without B- or type mismatch, start a new entity for robustness
                if current_entity:
                    processed_entities.append(current_entity)
                current_entity = {"name": word_text, "type": entity_type_raw}
        else: # Handle 'O' (outside) tag or other unexpected prefixes by ending current entity
            if current_entity:
                processed_entities.append(current_entity)
            current_entity = {} # Reset current entity

    if current_entity: # Add the last entity if it exists after the loop
        processed_entities.append(current_entity)

    # Filter for PERSON and ORG and normalize type names to match original SpaCy output types
    final_entities = []
    for entity in processed_entities:
        if entity["type"] == "PER":
            final_entities.append({"name": entity["name"], "type": "PERSON"})
        elif entity["type"] == "ORG":
            final_entities.append({"name": entity["name"], "type": "ORG"})
    return final_entities

def run_extraction():
    driver = GraphDatabase.driver(URI, auth=AUTH)

    # 1. Create Constraints (Crucial for performance)
    with driver.session() as session:
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")
        print("✅ Constraints Verified.")

    # 2. Process in Batches
    batch_size = 500
    while True:
        with driver.session() as session:
            # Fetch documents that don't have a 'processed' flag yet
            docs = session.run("""
                MATCH (d:Document)
                WHERE d.processed IS NULL AND d.content IS NOT NULL
                RETURN d.doc_name AS name, d.content AS text
                LIMIT $limit
            """, limit=batch_size)

            data_list = list(docs)
            if not data_list:
                print("🏁 All documents processed!")
                break

            payload = []
            for record in tqdm.tqdm(data_list, desc="Processing Entities"):
                doc_name = record["name"]
                text = record["text"]

                # Hugging Face NER Magic
                # Cap text per file to avoid memory spikes. The Hugging Face pipeline
                # typically handles chunking for models with max_sequence_length internally.
                chunk_text = text[:50000] # General cap to limit input size
                ner_results = ner_pipeline(chunk_text)

                entities = process_hf_ner_output(ner_results)

                payload.append({"doc_name": doc_name, "entities": entities})

            # 3. Batch Write back to the Neo4j
            session.run("""
                UNWIND $payload AS item
                MATCH (d:Document {doc_name: item.doc_name})
                SET d.processed = true
                WITH d, item
                UNWIND item.entities AS ent
                MERGE (e:Entity {name: ent.name})
                ON CREATE SET e.type = ent.type
                MERGE (d)-[:MENTIONS]->(e)
            """, payload=payload)

    driver.close()

run_extraction()

✅ Constraints Verified.


Processing Entities: 100%|██████████| 500/500 [00:12<00:00, 41.00it/s]
ERROR:neo4j.io:[#A99E]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687))): OSError('No data')


KeyboardInterrupt: 

# Task
**Reasoning:**
The `run_extraction` function failed again with a `ServiceUnavailable` error, indicating that the `cloudflared` tunnel connection to Neo4j was lost during processing. This requires re-establishing the tunnel. After the tunnel is confirmed to be running, I will re-execute the `run_extraction` function. The previous output shows that the `tqdm` progress bar is already providing visibility into entity processing, and the Neo4j queries (`MERGE` for entities/relationships, and `WHERE d.processed IS NULL` for documents) inherently handle deduplication as required by the plan.

**Plan:**
1. Re-run cell `eae0da0f` to re-establish the `cloudflared` tunnel connection.
2. Re-run cell `14afb956` to execute the `run_extraction` function again, allowing it to connect to Neo4j and complete the entity extraction process with the improved visibility and deduplication logic.

## Re-establish Cloudflared Tunnel

### Subtask:
Re-establish the `cloudflared` tunnel connection to Neo4j.


**Reasoning**:
To re-establish the `cloudflared` tunnel connection to Neo4j, I need to re-run the code in cell `eae0da0f` which initiates the tunnel.



In [ ]:
# Download and install cloudflared
!curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared.deb

# Run the bridge in the background
# This maps 'localhost:7687' in Kaggle -> 'bolt.ahhhh.men' -> Your Local DB
import subprocess

# REPLACE THESE with your Service Token if Access is enabled.
# If Access is disabled/bypassed, you can remove the --id and --secret flags.
cmd = [
    "cloudflared", "access", "tcp",
    "--hostname", "bolt.ahhhh.men",
    "--url", "localhost:7687",
    "--id", "facec3bfaf0d896f095f368020071d7a.access",      # Remove if Access is disabled
    "--secret", "7478557ddccc780cd3322528be98e3f8e811c63ab774cf0f239985d89a2c988a" # Remove if Access is disabled
]

# Run as a background process
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Bridge started on localhost:7687")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 19.1M  100 19.1M    0     0  20.8M      0 --:--:-- --:--:-- --:--:-- 20.8M
(Reading database ... 121693 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.1.2) over (2026.1.2) ...
Setting up cloudflared (2026.1.2) ...
Processing triggers for man-db (2.10.2-1) ...
Bridge started on localhost:7687


**Reasoning**:
Since the `cloudflared` tunnel has been re-established, I will now re-execute the `run_extraction` function, which has already been updated to use the Hugging Face NER pipeline. This will allow it to connect to Neo4j and process the documents.



In [ ]:
;from neo4j import GraphDatabase
import tqdm # For a progress bar

# Assume ner_pipeline is globally available from previous cell execution
# URI and AUTH are also globally available

# Helper function to process Hugging Face NER output
def process_hf_ner_output(ner_results):
    processed_entities = []
    current_entity = {}

    for ent in ner_results:
        # Extract base entity type (e.g., PER, ORG) from tags like 'B-PER', 'I-ORG'
        entity_type_raw = ent['entity'][2:]
        prefix = ent['entity'][:2]

        # Handle '##' subword tokens by removing them for cleaner entity names
        word_text = ent['word'].replace('##', '')

        if prefix == 'B-':
            if current_entity: # If there was a previous entity, add it to the list
                processed_entities.append(current_entity)
            current_entity = {"name": word_text, "type": entity_type_raw}
        elif prefix == 'I-':
            # Extend current entity if it exists and type matches
            if current_entity and current_entity["type"] == entity_type_raw:
                current_entity["name"] += " " + word_text
            else: # If I- without B- or type mismatch, start a new entity for robustness
                if current_entity:
                    processed_entities.append(current_entity)
                current_entity = {"name": word_text, "type": entity_type_raw}
        else: # Handle 'O' (outside) tag or other unexpected prefixes by ending current entity
            if current_entity:
                processed_entities.append(current_entity)
            current_entity = {} # Reset current entity

    if current_entity: # Add the last entity if it exists after the loop
        processed_entities.append(current_entity)

    # Filter for PERSON and ORG and normalize type names to match original SpaCy output types
    final_entities = []
    for entity in processed_entities:
        if entity["type"] == "PER":
            final_entities.append({"name": entity["name"], "type": "PERSON"})
        elif entity["type"] == "ORG":
            final_entities.append({"name": entity["name"], "type": "ORG"})
    return final_entities

def run_extraction():
    driver = GraphDatabase.driver(URI, auth=AUTH)

    # 1. Create Constraints (Crucial for performance)
    with driver.session() as session:
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")
        print("✅ Constraints Verified.")

    # 2. Process in Batches
    batch_size = 500
    while True:
        with driver.session() as session:
            # Fetch documents that don't have a 'processed' flag yet
            docs = session.run("""
                MATCH (d:Document)
                WHERE d.processed IS NULL AND d.content IS NOT NULL
                RETURN d.doc_name AS name, d.content AS text
                LIMIT $limit
            """, limit=batch_size)

            data_list = list(docs)
            if not data_list:
                print("🏁 All documents processed!")
                break

            payload = []
            for record in tqdm.tqdm(data_list, desc="Processing Entities"):
                doc_name = record["name"]
                text = record["text"]

                # Hugging Face NER Magic
                # Cap text per file to avoid memory spikes. The Hugging Face pipeline
                # typically handles chunking for models with max_sequence_length internally.
                chunk_text = text[:50000] # General cap to limit input size
                ner_results = ner_pipeline(chunk_text)

                entities = process_hf_ner_output(ner_results)

                payload.append({"doc_name": doc_name, "entities": entities})

            # 3. Batch Write back to the Neo4j
            session.run("""
                UNWIND $payload AS item
                MATCH (d:Document {doc_name: item.doc_name})
                SET d.processed = true
                WITH d, item
                UNWIND item.entities AS ent
                MERGE (e:Entity {name: ent.name})
                ON CREATE SET e.type = ent.type
                MERGE (d)-[:MENTIONS]->(e)
            """, payload=payload)

    driver.close()

run_extraction()

✅ Constraints Verified.


Processing Entities: 100%|██████████| 500/500 [00:11<00:00, 42.07it/s]
ERROR:neo4j.io:[#9810]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687))): OSError('No data')


KeyboardInterrupt: 

# Task
The agent has confirmed the previous plan was approved. The current task is to perform data cleaning on the Neo4j graph, focusing on identifying and merging duplicate entities, removing irrelevant ones, and correcting type mismatches.

**Plan:**

*   **Re-establish Cloudflared Tunnel:** Ensure the `cloudflared` tunnel is active and connected to the Neo4j database.
*   **Identify Potential Entity Duplicates:** Query existing `Entity` nodes, apply string similarity metrics to identify groups of likely duplicates, and generate a list of candidate merges.
*   **Generate and Execute Entity Merge Operations:** Based on the identified duplicates, construct and execute Cypher queries to merge entities, re-pointing relationships to a primary node and deleting alias nodes.
*   **Identify and Remove Irrelevant Entities:** Define criteria for irrelevant entities (e.g., short names, stopwords), query Neo4j for matches, and remove them along with their relationships.
*   **Review and Correct Entity Type Mismatches:** Identify entities with potentially incorrect `type` properties (e.g., `PERSON`, `ORG`) and generate Cypher queries to update them.
*   **Final Task:** Summarize the performed data cleaning operations, including counts of merged, removed, and corrected entities.

## Re-establish Cloudflared Tunnel

### Subtask:
Ensure the `cloudflared` tunnel is active and connected to the Neo4j database.


## Identify Potential Entity Duplicates

### Subtask:
Query existing `Entity` nodes from Neo4j, apply string similarity metrics to identify groups of likely duplicates, and generate a list of candidate merges.


**Reasoning**:
First, I need to ensure that the `fuzzywuzzy` library is installed, as it is a prerequisite for calculating string similarity metrics. I will use a `pip install` command for this.



In [ ]:
import sys
!{sys.executable} -m pip install fuzzywuzzy[speedup]
!{sys.executable} -m pip install python-Levenshtein # For faster processing with fuzzywuzzy
print("Installed fuzzywuzzy and python-Levenshtein.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.5 MB/s eta 0:00:00
Installed fuzzywuzzy and python-Levenshtein.


**Reasoning**:
Now that `fuzzywuzzy` is installed, I will import the necessary libraries, define connection parameters, and implement the `fetch_entity_names` and `find_potential_duplicates` functions as instructed. Finally, I will execute these functions to query the Neo4j database, identify potential duplicate entities based on string similarity, and print the results.



**Reasoning**:
The previous cell failed with a `ModuleNotFoundError` for the `neo4j` library. To resolve this, I will explicitly reinstall the `neo4j` library to ensure it is available in the environment.



In [ ]:
import sys
!{sys.executable} -m pip install neo4j
print("neo4j library ensured to be installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 6.8 MB/s eta 0:00:00
neo4j library ensured to be installed.


**Reasoning**:
Since the `neo4j` library has been successfully re-installed, I will now re-execute the code to fetch entity names from the Neo4j database and identify potential duplicates. This will ensure that the necessary library is available and the process can continue as intended.



In [ ]:
from neo4j import GraphDatabase
from fuzzywuzzy import fuzz

# Connection Config (globally available from previous cells)
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "jaspeR154$")

def fetch_entity_names():
    driver = GraphDatabase.driver(URI, auth=AUTH)
    entity_names = []
    with driver.session() as session:
        result = session.run("MATCH (e:Entity) RETURN e.name AS name")
        for record in result:
            entity_names.append(record["name"])
    driver.close()
    return entity_names

def find_potential_duplicates(entity_names, similarity_threshold=85):
    potential_duplicates = []
    # Use a set to keep track of names already processed as part of a group
    processed_names = set()

    # Convert names to lowercase for case-insensitive comparison
    lower_entity_names = {name.lower(): name for name in entity_names}
    unique_lower_names = list(lower_entity_names.keys())

    for i in range(len(unique_lower_names)):
        name1 = unique_lower_names[i]
        if name1 in processed_names: # Skip if already grouped
            continue

        current_group = {lower_entity_names[name1]}
        for j in range(i + 1, len(unique_lower_names)):
            name2 = unique_lower_names[j]

            # Use token_set_ratio for better handling of word order and partial matches
            similarity = fuzz.token_set_ratio(name1, name2)

            if similarity >= similarity_threshold:
                current_group.add(lower_entity_names[name2])

        if len(current_group) > 1:
            potential_duplicates.append(list(current_group))
            processed_names.update([n.lower() for n in current_group])

    return potential_duplicates

print("Fetching entity names from Neo4j...")
all_entity_names = fetch_entity_names()
print(f"Found {len(all_entity_names)} unique entity names.")

print("Identifying potential duplicate entities...")
duplicates = find_potential_duplicates(all_entity_names, similarity_threshold=85)

if duplicates:
    print("Potential duplicate entity groups:")
    for group in duplicates:
        print(f"  - {group}")
else:
    print("No significant potential duplicate entities found.")

Fetching entity names from Neo4j...


ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [Errno 111] Connection refused)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)

**Reasoning**:
The previous attempt to fetch entity names failed with a `ServiceUnavailable` error, indicating that the connection to the Neo4j database via the `cloudflared` tunnel was lost. I need to re-establish this tunnel to allow the Python script to connect to Neo4j.



In [ ]:
# Download and install cloudflared
!curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared.deb

# Run the bridge in the background
# This maps 'localhost:7687' in Kaggle -> 'bolt.ahhhh.men' -> Your Local DB
import subprocess

# REPLACE THESE with your Service Token if Access is enabled.
# If Access is disabled/bypassed, you can remove the --id and --secret flags.
cmd = [
    "cloudflared", "access", "tcp",
    "--hostname", "bolt.ahhhh.men",
    "--url", "localhost:7687",
    "--id", "facec3bfaf0d896f095f368020071d7a.access",      # Remove if Access is disabled
    "--secret", "7478557ddccc780cd3322528be98e3f8e811c63ab774cf0f239985d89a2c988a" # Remove if Access is disabled
]

# Run as a background process
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Bridge started on localhost:7687")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 19.1M  100 19.1M    0     0  17.7M      0  0:00:01  0:00:01 --:--:-- 53.7M
Selecting previously unselected package cloudflared.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.1.2) ...
Setting up cloudflared (2026.1.2) ...
Processing triggers for man-db (2.10.2-1) ...
Bridge started on localhost:7687


**Reasoning**:
Since the `cloudflared` tunnel has been successfully re-established, I will now re-execute the code to fetch entity names from the Neo4j database and identify potential duplicates. This will ensure that the necessary library is available and the process can continue as intended.



In [ ]:
from neo4j import GraphDatabase
from fuzzywuzzy import fuzz

# Connection Config (globally available from previous cells)
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "jaspeR154$")

def fetch_entity_names():
    driver = GraphDatabase.driver(URI, auth=AUTH)
    entity_names = []
    with driver.session() as session:
        result = session.run("MATCH (e:Entity) RETURN e.name AS name")
        for record in result:
            entity_names.append(record["name"])
    driver.close()
    return entity_names

def find_potential_duplicates(entity_names, similarity_threshold=85):
    potential_duplicates = []
    # Use a set to keep track of names already processed as part of a group
    processed_names = set()

    # Convert names to lowercase for case-insensitive comparison
    lower_entity_names = {name.lower(): name for name in entity_names}
    unique_lower_names = list(lower_entity_names.keys())

    for i in range(len(unique_lower_names)):
        name1 = unique_lower_names[i]
        if name1 in processed_names: # Skip if already grouped
            continue

        current_group = {lower_entity_names[name1]}
        for j in range(i + 1, len(unique_lower_names)):
            name2 = unique_lower_names[j]

            # Use token_set_ratio for better handling of word order and partial matches
            similarity = fuzz.token_set_ratio(name1, name2)

            if similarity >= similarity_threshold:
                current_group.add(lower_entity_names[name2])

        if len(current_group) > 1:
            potential_duplicates.append(list(current_group))
            processed_names.update([n.lower() for n in current_group])

    return potential_duplicates

print("Fetching entity names from Neo4j...")
all_entity_names = fetch_entity_names()
print(f"Found {len(all_entity_names)} unique entity names.")

print("Identifying potential duplicate entities...")
duplicates = find_potential_duplicates(all_entity_names, similarity_threshold=85)

if duplicates:
    print("Potential duplicate entity groups:")
    for group in duplicates:
        print(f"  - {group}")
else:
    print("No significant potential duplicate entities found.")

Fetching entity names from Neo4j...
Found 80104 unique entity names.
Identifying potential duplicate entities...


KeyboardInterrupt: 